In [ ]:
import pandas as pd
df=pd.read_csv("data/IMDB Dataset.csv")

In [7]:
df.drop_duplicates(inplace=True)

In [9]:
##pre-processing
##1.convert to lowercase
df["review"]=df["review"].str.lower()

In [10]:
##2, removing urls
import re ##re(regular expression) is a popular library of python used to identify patterns in exp

def remove_urls(text):
  text=re.sub(r"http\S+","",text)
  return text

df["review"]=df["review"].apply(remove_urls)

In [11]:
## removing punctuation

def remove_punc(text):
  text=re.sub(r"[^A-Za-z0-9\s]","",text)
  return text

df["review"]=df["review"].apply(remove_punc)

In [12]:
## 4. remove html tags

def remove_tags(text):
  text=re.sub(r"<.*?>","",text)
  return text
df["review"]=df["review"].apply(remove_tags)

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
def remove_stopwords(text):
  tokens=word_tokenize(text)
  stop_words=stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text=text.replace(word,"")
  return text

df["review"]=df["review"].apply(remove_stopwords)

In [ ]:
df.head()

In [17]:
## stemming
from nltk.stem import PorterStemmer

def stemming(text):
  ps=PorterStemmer()
  stemmed_words=[]

  tokens=word_tokenize(text)

  for token in tokens:
    stemm_token=ps.stem(token)
    stemmed_words.append(stemm_token)

  return " ".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

In [18]:
##7.encoding

from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

y=df["sentiment"]

In [19]:
## 8.vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])

In [21]:
## dataset and dataloader

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [22]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [23]:
import torch
from torch.utils.data import TensorDataset,DataLoader

train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)


In [24]:
train_loader=DataLoader(train_set,batch_size=64,shuffle=True)
test_loader=DataLoader(test_set,batch_size=64,shuffle=True)

In [25]:
# build Rnn
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
  def __init__(self,input_size,hidden_size=128,num_layers=1):
    super().__init__()
    self.hidden_size=hidden_size
    self.num_layers=num_layers
    #Rnn layer
    self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

    #full conected layer
    self.fc=nn.Linear(hidden_size,1)

  def forward(self,x):
    # optional ->shape(num of layers,batch_size,hidden_size)
    h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
    out,_=self.rnn(x,h0)


    out=self.fc(out[:,-1,:])
    return out

In [26]:
input_size=X_train.shape[1]


model=RNN(input_size)
criteria=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        xb = xb.unsqueeze(1)

        outputs = model(xb)
        outputs = torch.sigmoid(outputs.squeeze())

        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch:{epoch + 1} loss:{loss.item()}")

In [ ]:
model.eval()
with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        outputs = model(xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy:{correct_vals / total_vals}")